# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Lane: Refresh / Content Opportunity Scoring.**
**Question:** which content pages should an SEO/content team review first for a possible
refresh, given limited review time? **Decision supported:** a ranked list, reviewed top-down by
a human — not an automated determination that any specific page needs changing.


In [ ]:
print("Lane: Refresh / Content Opportunity Scoring")
print("Decision: which content pages should a human review first?")
print("Output: a ranked list, one row per content page, with one reason code each.")


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Tables used:** `fact_content_daily_performance` (content-level, monthly — powers the baseline
and signal audit) and `fact_content_query_90d` (query-level, fixed 90-day window — powers the
feature/leakage contract and the model). Both joined to `dim_content` for static SEO/content
metadata. **Window:** a single fixed 90-day snapshot (`2026-04-02` to `2026-06-30`, verified
directly from the data, not assumed) with two named 30-day sub-windows, `prev30` and `last30`.
**Excluded, and why:** the `_90d` aggregates (numerically contain the `last30` label window —
leakage), `provider_used`/`model_used` (backend detail), `content_updated_date`/
`last_optimized_date` (reflect FlyRank's own past decisions, not new signal). Full reasoning in
`w03_data_contract.ipynb` and `w03_feature_leakage_check.ipynb`. All identifiers are anonymized
hashes — no client names, URLs, or raw queries appear anywhere in this project.


In [ ]:
print("Data: fact_content_daily_performance + fact_content_query_90d + dim_content")
print("Window: fixed 90 days (2026-04-02 to 2026-06-30), prev30/last30 sub-windows")
print("Excluded: *_90d aggregates, provider/model flags, content_updated_date, last_optimized_date")


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label:** `ctr_gap` — a page's actual CTR subtracted from the typical CTR for its position
bucket, at content-page grain (for the baseline) and query grain (for the feature contract).
**Baseline (ML-07):** one signal (`CTR_GAP_VS_POSITION`, confirmed in the signal audit,
independently backed by FlyRank's own published research — Finding #3), one reason code, one
action (`REWRITE_TITLE_META`), ranked by `ctr_gap × impressions`.
**Model (ML-08):** gradient-boosted trees, trained on `prev30` + static SEO/content features
only (never this month's own GSC numbers), evaluated with `GroupKFold` on `client_hash_id` so
no client appears in both train and test.
**Validation (ML-09):** grouped split vs. a naive random split run side-by-side to show the
inflation a wrong split would have hidden; the same three-part leakage hunt re-run on the final
feature set; two published-paper findings checked against this project's own signal audit —
one agreed (CTR vs. position), one didn't (staleness/freshness), and that disagreement is
carried forward rather than smoothed over.


In [ ]:
print("Label: ctr_gap (position-bucket-relative CTR gap)")
print("Baseline: single-signal rule, CTR_GAP_VS_POSITION")
print("Model: gradient-boosted trees, prev30 + static features only, GroupKFold by client")
print("Validation: grouped vs naive split, re-run leakage hunt, cross-check vs published paper")


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

**NOT FILLED IN — do not write numbers here until `w05_model.ipynb` and
`w06_validation_audit.ipynb` have actually been run.** This section needs two real, printed
numbers: the grouped-split Precision@50 for the model, and for the `prev30`-based baseline,
from ML-08's Section 3 output. Copy them here verbatim, plus ML-09's honest-vs-naive-split
comparison, once you have them. If the model does not beat the baseline, **say that plainly**
— a well-argued negative result is a legitimate capstone finding, not a failure to write up.


In [ ]:
import pandas as pd
try:
    print("Paste the real results table from w05_model.ipynb Section 3 here, e.g.:")
    print(pd.DataFrame({"method": ["baseline (prev30 CTR gap)", "model (gradient boosting)"],
                         "precision_at_50": ["<fill in from ML-08>", "<fill in from ML-08>"]}))
except Exception as e:
    print(e)


## 5. Limitations

*What this work cannot claim.*

- `ctr_gap` is an observed diagnostic, not proof a rewrite will help — no outcome-of-rewrite
  data exists in this warehouse to confirm the fix works.
- The baseline's impression-weighted score structurally favors high-traffic clients/content;
  a smaller client with a proportionally worse gap can rank lower purely on volume (ML-07,
  weak-picks section).
- Low/zero CTR can come from SERP features or branded-query intent that a title rewrite cannot
  fix — flagged concretely in ML-07's top-20 review, not resolved by this system.
- This project's own staleness signal (FALSE) disagrees with FlyRank's own published research
  (freshness "CONFIRMED" as a strong lever) — different portfolios, different populations,
  genuinely unresolved rather than picked a side on.
- Single fixed 90-day snapshot — no way to confirm these patterns hold across seasons or over
  a longer horizon.


In [ ]:
print("Limitations recorded above are carried forward from ML-07, ML-06, and ML-09 — not new claims.")


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

The action playbook (`w07_action_playbook.ipynb`) is this section's real content: the ranked
queue from `work/outputs/baseline_action_score.csv` (or the model's ranking, if ML-09 shows it
wins), one reason code per row, paired with the no-go list (never auto-publish a rewrite) and
the retrain/monitoring triggers (watch for drift in the position-bucket CTR curve month to
month).


In [ ]:
import pandas as pd
top20 = pd.read_csv("work/outputs/top20_for_paper.csv")
print(top20[["client_hash_id","content_hash_id","action_score","reason_code","action"]])


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Embeds the two artifacts `w07_action_playbook.ipynb` already generated:
`work/outputs/top20_for_paper.csv` (the reviewed top-20 table) and
`work/outputs/ctr_by_position_bucket.png` (the CTR-by-position curve the whole baseline rests
on). If ML-08/09 are complete by the time this is finalized, also embed the Precision@50
comparison chart — code for that is in ML-08 Section 3's `results` DataFrame; a simple bar
chart of `model_precision_at_50` vs `baseline_precision_at_50` is enough, don't over-design it.


In [ ]:
from IPython.display import Image
print("Artifacts for the paper: top20_for_paper.csv, ctr_by_position_bucket.png")
# Image("work/outputs/ctr_by_position_bucket.png")  # uncomment once the file exists locally


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.